### Phase A: Data Engineering & Graph Extraction

**Methodology:**
This notebook is responsible for constructing the foundational data layer. We utilize `osmnx` to query OpenStreetMap and extract the actual road network for Bengaluru's Outer Ring Road (ORR). By operating on real geospatial bounds rather than a synthetic grid, we ensure the simulation's transit graph mirrors real-world traffic topologies. A synthetic fallback is also provided to generate deterministic arrival matrices and traffic delay maps when live APIs are unavailable.

In [ ]:
# Install dependencies
!pip install pandas numpy osmnx faker scipy kaggle

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.stats import poisson
import osmnx as ox
import kaggle

print("Libraries imported successfully.")

In [ ]:
# Cell 2: OSM Graph Extraction
print("Extracting OSM graph for Outer Ring Road, Bengaluru...")
try:
    # Note: Using 'Outer Ring Road, Bengaluru, India' as place query might pull the entire country if Nominatim fails.
    # We use a lightweight bounding box around a segment of ORR (Agara/HSR Layout) to ensure rapid prototyping.
    graph = ox.graph_from_bbox(bbox=(77.63, 12.92, 77.65, 12.93), network_type='drive')
    nodes, edges = ox.graph_to_gdfs(graph)
    print(f"Graph extracted with {len(nodes)} nodes and {len(edges)} edges.")
except Exception as e:
    print(f"Extraction failed: {e}")
    graph, nodes, edges = None, [], []

In [ ]:
# Cell 3: The Synthetic Fallback (CRITICAL)
num_stops = 30
time_steps = 120

# Poisson-distributed passenger arrival rates (λ=8 peak, λ=2 off-peak)
peak_arrivals = poisson.rvs(mu=8, size=(num_stops, 60))
offpeak_arrivals = poisson.rvs(mu=2, size=(num_stops, 60))
arrival_rates = np.hstack([peak_arrivals, offpeak_arrivals])

# Normally-distributed traffic delay matrices simulating Bengaluru ORR traffic
num_segments = num_stops - 1
delays_matrix = np.abs(np.random.normal(loc=45.0, scale=15.0, size=(num_segments, time_steps)))

# Baseline economic variables
economic_baselines = {
    'wait_cost_per_min': 1.67,
    'fuel_cost_per_min': 0.83,
    'ticket_price': 15.0
}

In [ ]:
# Cell 4: Export to Dictionary
bengaluru_simulation_data = {
    'graph': graph,
    'graph_nodes_gdf': nodes,
    'graph_edges_gdf': edges,
    'arrival_rates': arrival_rates,
    'delays_matrix': delays_matrix,
    'economic_baselines': economic_baselines,
    'metadata': {
        'num_stops': num_stops,
        'time_steps': time_steps,
        'description': "Synthetic Bengaluru ORR Data for ECO-SYNC"
    }
}

os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/bengaluru_simulation_data.pkl"
with open(output_path, 'wb') as f:
    pickle.dump(bengaluru_simulation_data, f)

print(f"Data successfully generated and saved to {output_path}")
print(f"Data shape summary:")
print(f" - Graph nodes: {len(nodes)}")
print(f" - Graph edges: {len(edges)}")
print(f" - Arrival rates matrix shape: {arrival_rates.shape}")
print(f" - Delays matrix shape: {delays_matrix.shape}")